# Notebook para transferência de arquivos de um bucket para outro
Autor: Suzane Carol de Lima  
Entrega: 05/03/26
## Objetivo:
Notebook de transferência dos dados do bucket analise-dados para o bucket do PDI digital

# Importações de bibliotecas

In [6]:
import json
import boto3
import time
from botocore.exceptions import ClientError
from boto3.resources.base import ServiceResource
from botocore.paginate import Paginator
from typing import Set, Tuple


# Declaração de constantes

In [4]:
COPIED_FILE = 'copiados.txt'
LOG_FILE = 'logs.txt'

BUCKET_ORIGEM = 'analise-dados'
BUCKET_DESTINO = 's3-ma-analytics-259755696523-sa-east-1'
PREFIXO_ORIGEM = 'projeto-ia-submarina/ia-descomissionamento/OKR_14 - Merge_Shapefiles/85520-6000685899/shape_classes_completo/'
PREFIXO_DESTINO = 'EADigital/PDI_Agulha/ID_85520_OS_6000685899_OL4_PAG3_PAG1_B/shapefiles_agrupados_OS/'

# Funções Necessárias

In [7]:
def carregar_chaves_copiadas(caminho: str) -> Set[str]:
    """
    Carrega as chaves já copiadas de um arquivo texto, retornando um conjunto de strings.

    Parâmetros:
        caminho (str): Caminho do arquivo contendo as chaves copiadas.

    Retorno:
        Set[str]: Conjunto das chaves copiadas.
    """
    try:
        with open(caminho, 'r') as f:
            return set(linha.strip() for linha in f if linha.strip())
    except:
        return set()

def salvar_chave_copiada(caminho: str, chave: str) -> None:
    """
    Salva uma chave copiada no arquivo especificado.

    Parâmetros:
        caminho (str): Caminho do arquivo onde será salva a chave.
        chave (str): Chave a ser salva.

    Retorno:
        None
    """
    with open(caminho, 'a') as f:
        f.write(f'{chave}\n')

def salvar_log(caminho: str, mensagem: str) -> None:
    """
    Salva uma mensagem de log com timestamp no arquivo especificado.

    Parâmetros:
        caminho (str): Caminho do arquivo de log.
        mensagem (str): Mensagem a ser registrada.

    Retorno:
        None
    """
    ts = time.strftime('%Y-%m-%d %H:%M:%S')
    with open(caminho, 'a') as f:
        f.write(f'[{ts}] {mensagem}\n')

def carregar_configuracao_aws(caminho_config: str) -> dict:
    """
    Carrega o arquivo de configuração AWS em formato JSON.

    Parâmetros:
        caminho_config (str): Caminho para o arquivo JSON de configuração.

    Retorno:
        dict: Dicionário com as configurações carregadas.
    """
    with open(caminho_config) as f:
        config = json.load(f)
    return config

def criar_clientes_s3(config: dict)-> Tuple:
    """
    Cria e retorna os clientes S3 de origem e destino usando as configurações fornecidas.

    Parâmetros:
        config (dict): Dicionário com as configurações de acesso AWS para 'analise-dados' (origem)
                       e 'pdi-digital' (destino), podendo conter 'session_token' para destino.

    Retorno:
        tuple: (source_s3, dest_s3)
            source_s3: Cliente S3 da origem ('analise-dados')
            dest_s3: Cliente S3 do destino ('pdi-digital')
    """
    source_s3 = boto3.client(
        's3',
        aws_access_key_id=config['analise-dados']['aws_access_key_id'],
        aws_secret_access_key=config['analise-dados']['aws_secret_access_key']
    )

    if 'session_token' in config['pdi-digital']:
        print("vou usar")
        dest_s3 = boto3.client(
            's3',
            aws_access_key_id=config['pdi-digital']['aws_access_key_id'],
            aws_secret_access_key=config['pdi-digital']['aws_secret_access_key'],
            aws_session_token=config['pdi-digital']['session_token']
        )
    else:
        dest_s3 = boto3.client(
            's3',
            aws_access_key_id=config['pdi-digital']['aws_access_key_id'],
            aws_secret_access_key=config['pdi-digital']['aws_secret_access_key']
        )
    return source_s3, dest_s3

def obter_paginador_list_objects(s3) -> Paginator:
    """
    Retorna o paginador para a operação 'list_objects_v2' no S3.

    Parâmetros:
        s3: Cliente S3 autenticado.

    Retorno:
        Paginador configurado para 'list_objects_v2'.
    """
    return s3.get_paginator('list_objects_v2')

def listar_objetos_s3(paginador, bucket_origem: str, prefix_origem: str)-> list:
    """
    Retorna uma lista com as chaves (keys) de todos os objetos encontrados em um bucket S3 a partir de um prefixo.

    Parâmetros:
        paginador: Paginador do S3 configurado para 'list_objects_v2'.
        bucket_origem (str): Nome do bucket de origem.
        prefix_origem (str): Prefixo base para busca dos objetos.

    Retorno:
        Lista de strings contendo as chaves dos objetos encontrados.
    """
    objetos_encontrados = []
    for pagina in paginador.paginate(Bucket=bucket_origem, Prefix=prefix_origem):
        if 'Contents' in pagina:
            for obj in pagina['Contents']:
                objetos_encontrados.append(obj['Key'])
    print('Objetos encontrados:', len(objetos_encontrados))
    return objetos_encontrados

def copiar_arquivos_s3(
    objetos_encontrados: list, 
    prefix_origem: str, 
    prefix_destino: str, 
    bucket_origem: str, 
    bucket_destino: str, 
    source_s3, 
    dest_s3, 
    copied_file: str, 
    log_file: str
)-> None:
    """
    Copia objetos de um bucket S3 de origem para um bucket S3 de destino, evitando duplicidades e registrando logs.

    Parâmetros:
        objetos_encontrados (list): Lista de chaves S3 a serem copiadas.
        prefix_origem (str): Prefixo de origem das chaves.
        prefix_destino (str): Prefixo de destino das chaves.
        bucket_origem (str): Nome do bucket de origem.
        bucket_destino (str): Nome do bucket de destino.
        source_s3: Cliente S3 de origem.
        dest_s3: Cliente S3 de destino.
        copied_file (str): Caminho do arquivo de controle de chaves copiadas.
        log_file (str): Caminho do arquivo de log.

    Retorno:
        None
    """   

    copied_keys = carregar_chaves_copiadas(copied_file)
    salvar_log(log_file, f'Início da cópia de {len(objetos_encontrados)} arquivos.')

    for key in objetos_encontrados:
        if key.endswith('/') or key in copied_keys:
            continue
        new_key = key.replace(prefix_origem, prefix_destino, 1)
        try:
            obj = source_s3.get_object(Bucket=bucket_origem, Key=key)
            dest_s3.put_object(Bucket=bucket_destino, Key=new_key, Body=obj['Body'].read())
            salvar_chave_copiada(copied_file, key)
            print(f'Copiado: {key}')
        except ClientError as e:
            if "ExpiredToken" in str(e):
                msg = 'Cópia interrompida por expiração ou problema de autenticação.'
                print(msg)
                salvar_log(log_file, msg)
                salvar_log(log_file, 'Cópia finalizada por expiração ou problema de autenticação.')
                print('Cópia finalizada por expiração ou problema de autenticação.')
                break
            else:
                msg = f'Erro ao copiar {key}: {e.response["Error"]["Code"]} - {e.response["Error"]["Message"]}'
                print(msg)
                salvar_log(log_file, msg)
                salvar_log(log_file, 'Execução finalizada devido a erro.')
                print('Execução finalizada devido a erro.')
                break
        except Exception as e:
            msg = f'Erro inesperado ao copiar {key}: {e}'
            print(msg)
            salvar_log(log_file, msg)
            salvar_log(log_file, 'Execução finalizada devido a erro inesperado.')
            print('Execução finalizada devido a erro inesperado.')
            break
    else:
        salvar_log(log_file, 'Cópia finalizada: todos os arquivos foram enviados.')
        print('Cópia finalizada: todos os arquivos foram enviados.')

def transferir_arquivos_restantes(
    source_s3,
    dest_s3,
    bucket_origem: str,
    bucket_destino: str,
    prefix_origem: str,
    prefix_destino: str,
    objetos_encontrados: list,
    copied_file: str,
    log_file: str
) -> None:
    """
    Transfere apenas os arquivos restantes (não copiados) de um bucket S3 de origem para um bucket S3 de destino, 
    registrando logs do processo e lidando com possíveis erros de autenticação e falhas inesperadas.

    Parâmetros:
        source_s3: Cliente S3 de origem.
        dest_s3: Cliente S3 de destino.
        bucket_origem (str): Nome do bucket de origem.
        bucket_destino (str): Nome do bucket de destino.
        prefix_origem (str): Prefixo de origem dos arquivos.
        prefix_destino (str): Prefixo de destino dos arquivos.
        objetos_encontrados (list): Lista de chaves dos arquivos a serem verificados/copiedos.
        copied_file (str): Caminho do arquivo de controle de chaves já copiadas.
        log_file (str): Caminho do arquivo de log.
        
    Retorno:
        None
    """
    from botocore.exceptions import ClientError

    copied_keys = carregar_chaves_copiadas(copied_file)

    salvar_log(log_file, f'Início da cópia dos arquivos restantes ({len(objetos_encontrados) - len(copied_keys)} arquivos).')

    finalizou_por_token = False

    for key in objetos_encontrados:
        if key.endswith('/') or key in copied_keys:
            continue
        new_key = key.replace(prefix_origem, prefix_destino, 1)
        try:
            obj = source_s3.get_object(Bucket=bucket_origem, Key=key)
            dest_s3.put_object(Bucket=bucket_destino, Key=new_key, Body=obj['Body'].read())
            salvar_chave_copiada(copied_file, key)
            print(f'Copiado: {key}')
        except ClientError as e:
            if "ExpiredToken" in str(e):
                msg = 'Cópia interrompida por expiração ou problema de autenticação.'
                print(msg)
                salvar_log(log_file, msg)
                finalizou_por_token = True
                break
            else:
                msg = f'Erro ao copiar {key}: {e.response["Error"]["Code"]} - {e.response["Error"]["Message"]}'
                print(msg)
                salvar_log(log_file, msg)
                finalizou_por_token = True
                break
        except Exception as e:
            msg = f'Erro inesperado ao copiar {key}: {e}'
            print(msg)
            salvar_log(log_file, msg)
            finalizou_por_token = True
            break

    if finalizou_por_token:
        salvar_log(log_file, 'Cópia finalizada por erro ou expiração do token.')
        print('Cópia finalizada por erro ou expiração do token.')
    else:
        salvar_log(log_file, 'Cópia finalizada: todos os arquivos restantes foram enviados.')
        print('Cópia finalizada: todos os arquivos restantes foram enviados.')

# Configurando acesso ao S3 do bucket de origem dos dados e do bucket de destino dos dados

In [18]:
config = carregar_configuracao_aws('configuracao_aws/config.json')

In [19]:
source_s3, dest_s3 = criar_clientes_s3(config)

vou usar


# Identificação de objetos no bucket de origem

In [20]:
paginator = obter_paginador_list_objects(source_s3)

In [21]:
objetos_encontrados = listar_objetos_s3(paginador, BUCKET_ORIGEM, BUCKET_DESTINO)

Objetos encontrados: 45


# Transferência dos arquivos do bucket de origem para o bucket de destino

In [22]:
copiar_arquivos_s3(
    objetos_encontrados, 
    PREFIX_ORIGEM, 
    PREFIX_DESTINO, 
    BUCKET_ORIGEM, 
    BUCKET_DESTINO, 
    source_s3, 
    dest_s3, 
    COPIED_FILE, 
    LOG_FILE
)

Copiado: projeto-ia-submarina/ia-descomissionamento/OKR_14 - Merge_Shapefiles/85520-6000685899/shape_classes_completo/Track_bruto.cpg
Copiado: projeto-ia-submarina/ia-descomissionamento/OKR_14 - Merge_Shapefiles/85520-6000685899/shape_classes_completo/Track_bruto.dbf
Copiado: projeto-ia-submarina/ia-descomissionamento/OKR_14 - Merge_Shapefiles/85520-6000685899/shape_classes_completo/Track_bruto.prj
Copiado: projeto-ia-submarina/ia-descomissionamento/OKR_14 - Merge_Shapefiles/85520-6000685899/shape_classes_completo/Track_bruto.shp
Copiado: projeto-ia-submarina/ia-descomissionamento/OKR_14 - Merge_Shapefiles/85520-6000685899/shape_classes_completo/Track_bruto.shx
Copiado: projeto-ia-submarina/ia-descomissionamento/OKR_14 - Merge_Shapefiles/85520-6000685899/shape_classes_completo/duto_parcialmente_enterrado.cpg
Copiado: projeto-ia-submarina/ia-descomissionamento/OKR_14 - Merge_Shapefiles/85520-6000685899/shape_classes_completo/duto_parcialmente_enterrado.dbf
Copiado: projeto-ia-submarina/

# Transferência de arquivos faltantes (Session key expirou e não terminou de transferir todos os arquivos)

In [16]:
transferir_arquivos_restantes(
    source_s3,
    dest_s3,
    BUCKET_ORIGEM,
    BUCKET_DESTINO,
    PREFIX_ORIGEM,
    pREFIX_DESTINO,
    objetos_encontrados,
    COPIED_FILES,
    LOG_FILE)

Objetos encontrados: 13819
Copiado: projeto-ia-submarina/ia-descomissionamento/OKR_12_ApolloZ/85520-6000685899/6000685899_2024-03-30_041054_Ch4_32/json/Frame_1746_6000685899_2024-03-30_041054_Ch4_32_00_59.json
Copiado: projeto-ia-submarina/ia-descomissionamento/OKR_12_ApolloZ/85520-6000685899/6000685899_2024-03-30_041054_Ch4_32/json/Frame_18042_6000685899_2024-03-30_041054_Ch4_32_10_19.json
Copiado: projeto-ia-submarina/ia-descomissionamento/OKR_12_ApolloZ/85520-6000685899/6000685899_2024-03-30_041054_Ch4_32/json/Frame_18624_6000685899_2024-03-30_041054_Ch4_32_10_39.json
Copiado: projeto-ia-submarina/ia-descomissionamento/OKR_12_ApolloZ/85520-6000685899/6000685899_2024-03-30_041054_Ch4_32/json/Frame_19206_6000685899_2024-03-30_041054_Ch4_32_10_59.json
Copiado: projeto-ia-submarina/ia-descomissionamento/OKR_12_ApolloZ/85520-6000685899/6000685899_2024-03-30_041054_Ch4_32/json/Frame_19788_6000685899_2024-03-30_041054_Ch4_32_11_18.json
Copiado: projeto-ia-submarina/ia-descomissionamento/OK